# Starter notebook: compare a baseline model to the brains

This is a minimal, copy-me example of the whole pipeline:

1. **Download** a simple pretrained model (an ImageNet ResNet-18).
2. **Wrap** it in a small class (a subclass of `ModelAnalysisBase`) whose only real job is one method: `embedding()`.
3. **Run** it to build the model's RDM (see [CONTRIBUTING.md](../CONTRIBUTING.md) for what an RDM is).
4. **Compare** that RDM to the human fMRI (**Algonauts**) and macaque-neuron (**Triple-N**) data.

The fancier β-VAE version, with nicer figures, is in [beta_vae.ipynb](./beta_vae.ipynb).

## Running this locally vs. on Colab

This notebook works **either way** — the setup cell below auto-detects Colab.

**On your own computer (VS Code):** do the one-time setup in
[CONTRIBUTING.md](../CONTRIBUTING.md) first (`uv sync --all-extras`, plus a `.env` file with
`ALGONAUTS_DIR` and `TRIPLE_N_DIR`). Then just run the cells top to bottom.

**On Google Colab:** Colab installs this project **straight from GitHub**, so it can only see
code that is *already on GitHub*. So:

1. **Make a branch and push it** (see CONTRIBUTING.md → "Save your work and open a pull request"):
   ```bash
   git checkout -b my-branch
   git add . && git commit -m "my model"
   git push origin my-branch
   ```
2. In the setup cell below, set `BRANCH = "my-branch"` so Colab installs **your** code, not `main`.
3. Get the data onto Colab by mounting the Google Drive where you saved the
   [downloaded data](https://drive.google.com/drive/folders/1bWpBi-X9iN8x-HsuRwXDUX-GzELAK3sC):
   ```python
   from google.colab import drive
   drive.mount("/content/drive")
   ```
   then point the path variables in the config cell at those folders.

> ⚠️ **Every time you change code locally, you must `git push` again before Colab can see it.**

In [ ]:
# --- Works on Colab OR locally -------------------------------------------------
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# On Colab we install the project from GitHub. CHANGE THIS to your pushed branch!
BRANCH = "main"

if IN_COLAB:
    !pip install -q "git+https://github.com/lucas-nunn/PSM-NeuroAI-Final.git@{BRANCH}"
else:
    # Locally: load ../.env into the environment in case VS Code hasn't already
    # (this makes plain `jupyter` / command-line runs work too).
    env_path = Path("../.env")
    if env_path.exists():
        for line in env_path.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key, value)

print("Running on Colab" if IN_COLAB else "Running locally")

In [ ]:
# --- Tell the code where the data lives ----------------------------------------
if IN_COLAB:
    # >>> EDIT THESE to your mounted Google Drive folders (see the markdown above) <<<
    ALGONAUTS_DIR = "/content/drive/MyDrive/psm-data/algonauts/train"
    TRIPLE_N_DIR  = "/content/drive/MyDrive/psm-data/triple-N"
    # nsd_expdesign.mat ships with the repo; grab it from your branch on GitHub.
    if not os.path.exists("nsd_expdesign.mat"):
        !wget -q "https://raw.githubusercontent.com/lucas-nunn/PSM-NeuroAI-Final/{BRANCH}/nsd_expdesign.mat" -O nsd_expdesign.mat
    NSD_MAT = "nsd_expdesign.mat"
else:
    ALGONAUTS_DIR = os.environ["ALGONAUTS_DIR"]
    TRIPLE_N_DIR  = os.environ["TRIPLE_N_DIR"]
    NSD_MAT       = "../nsd_expdesign.mat"

print("ALGONAUTS_DIR =", ALGONAUTS_DIR)
print("TRIPLE_N_DIR  =", TRIPLE_N_DIR)

## Step 1 — Download a baseline model and wrap it

Our baseline is a **ResNet-18 pretrained on ImageNet** — a standard, supervised image
classifier, and a nice contrast to the *unsupervised* β-VAE. `torchvision` downloads the
weights automatically the first time you run this.

We wrap it in a subclass of `ModelAnalysisBase`. The base class already knows how to find the
stimulus images and turn features into an RDM; all we add is **`embedding()`**, which turns
**one** PIL image into a **1-D** feature vector. Here we grab the 512-number vector right
after the network's global average-pooling layer, using a PyTorch forward *hook*.

In [ ]:
import numpy as np
import torch
from torchvision.models import resnet18, ResNet18_Weights

from psm_final.analysis.model import ModelAnalysisBase


class ResNetAnalysis(ModelAnalysisBase):
    def __init__(self, triple_n_path, device=None):
        super().__init__(triple_n_path)            # lets .rdm() find the stimulus images

        self.device = torch.device(
            device or ("cuda" if torch.cuda.is_available() else "cpu")
        )

        # (1) Download the pretrained model + its matching preprocessing.
        weights = ResNet18_Weights.DEFAULT
        self.model = resnet18(weights=weights).to(self.device).eval()
        self.preprocess = weights.transforms()     # resize / crop / normalize a PIL image

        # (2) A hook that stashes the avg-pool layer's output on every forward pass.
        self._features = {}
        self.model.avgpool.register_forward_hook(
            lambda module, inp, out: self._features.__setitem__("z", out)
        )

    def embedding(self, image):
        # The base class hands us one PIL image at a time.
        x = self.preprocess(image.convert("RGB")).unsqueeze(0).to(self.device)
        with torch.no_grad():
            self.model(x)                          # run it; the hook fills self._features
        return self._features["z"].squeeze().cpu().numpy().ravel()   # 512-d vector

## Step 2 — Build the model's RDM

`.rdm()` runs your `embedding()` on all 1000 shared stimulus images and assembles the
results into a Representational Dissimilarity Matrix. (The first run also downloads the
weights. On a GPU this is seconds; on a laptop CPU, a couple of minutes.)

In [ ]:
import matplotlib.pyplot as plt
from scipy.spatial.distance import squareform

analysis = ResNetAnalysis(triple_n_path=TRIPLE_N_DIR)
model_rdm = analysis.rdm()          # condensed RDM over the 1000 shared images

plt.figure(figsize=(5, 4))
plt.imshow(squareform(model_rdm))
plt.title("ResNet-18 RDM (1000 shared images)")
plt.colorbar(label="1 - correlation")
plt.show()

## Step 3 — Compare to the macaque neurons (Triple-N)

Both `analysis.rdm()` and `triple_n.compute_rdm()` default to the **same 1000 images in the
same order**, so they line up automatically — no index bookkeeping needed. We score the match
with **Spearman correlation**: a higher `rho` means the model organizes images more like that
brain area does.

In [ ]:
from scipy.stats import spearmanr
from psm_final import TripleN

triple_n = TripleN(TRIPLE_N_DIR)

# A few areas along the monkey's visual hierarchy (early V1/V2/V4 -> high-level IT patches).
for area in ["V1", "V2", "V4", "Face", "Object"]:
    brain_rdm = triple_n.compute_rdm(area_label=area)
    rho, p = spearmanr(model_rdm, brain_rdm)
    print(f"ResNet-18  vs  Triple-N {area:7s}:  rho = {rho:+.3f}   (p = {p:.1e})")

## Step 4 — Compare to the human fMRI (Algonauts)

Algonauts needs a little alignment: it's indexed by NSD image id, and each of the 8 subjects
only saw *most* of the 1000 shared images. So we (1) find the images **every** subject saw,
(2) express that same set three ways — as NSD ids (for Algonauts), Triple-N `stim_index`
values, and 0-based positions (for the model) — and (3) build every RDM over that one shared,
aligned set. The helper methods do the heavy lifting.

In [ ]:
from psm_final import Algonauts, shared_stimuli

shared_ids = shared_stimuli(NSD_MAT)
algonauts = Algonauts(ALGONAUTS_DIR, shared_ids)
SUBJECTS = range(1, 9)

# (1) images present in EVERY subject's split ...
matched = [set(algonauts.shared_stimuli_indices(s, shared_ids)[0]) for s in SUBJECTS]
common = sorted(set.intersection(*matched))

# (2) ... that also have a Triple-N mapping, expressed three ways:
stim_idx = TripleN.nsd_to_stim_index(common)          # 1-based Triple-N stim_index (None if unmapped)
keep = [k for k, s in enumerate(stim_idx) if s is not None]
common_aligned = [common[k] for k in keep]            # NSD ids      -> Algonauts
stim_aligned   = [stim_idx[k] for k in keep]          # stim_index   -> Triple-N
model_indices  = np.array(stim_aligned) - 1           # 0-based rows -> model.rdm()
print(f"{len(common_aligned)} images shared across the model, the fMRI, and the neurons")

# (3) the model's RDM over exactly those images, in that order
model_rdm_aligned = analysis.rdm(indices=model_indices)

# Each Algonauts ROI RDM = average across the 8 subjects.
for roi in ["V1v", "hV4", "FFA-1", "EBA"]:
    per_subj = [algonauts.compute_rdm(subject=s, indices=common_aligned, roi=roi) for s in SUBJECTS]
    per_subj = [r for r in per_subj if r.std() > 0]   # skip subjects missing this ROI
    brain_rdm = np.vstack(per_subj).mean(axis=0)
    rho, p = spearmanr(model_rdm_aligned, brain_rdm)
    print(f"ResNet-18  vs  Algonauts {roi:7s}:  rho = {rho:+.3f}   (p = {p:.1e})")

## That's the whole pipeline 🎉

You just downloaded a model, wrapped it in ~15 lines, and scored it against a human brain and
a monkey brain over the same images.

**To contribute your own model:** copy the `ResNetAnalysis` class into a new file under
`src/psm_final/analysis/`, swap ResNet-18 for your AE / VAE / diffusion model, and change which
layer `embedding()` reads. Full instructions — plus the Git steps to open a pull request — are
in [CONTRIBUTING.md](../CONTRIBUTING.md). For richer comparison tables and saved figures, see
[beta_vae.ipynb](./beta_vae.ipynb).